# Matterhorn · a landscape in three dimensions

Real elevation data replaces the original paraboloid. The historic notebook and
export filenames are retained so existing links keep working.

Drag to orbit, scroll to zoom, switch to a north-up overhead view, or reveal
100 m contours. Horizontal and vertical axes both use kilometres, with **no
vertical exaggeration**. Colours encode elevation, not land cover or snow.

The 4,478 m summit is an independent reference; the gridded surface model smooths
sharp peaks and is not a surveyed climbing map. No elevation is artificially
raised to meet the reference marker.

## Data and reproducibility

[Copernicus DEM GLO-30](https://registry.opendata.aws/copernicus-dem/), AWS 2021
release, tiles N45/E007 and N46/E007, accessed 25 October 2026.
[COG format documentation](https://copernicus-dem-30m.s3.amazonaws.com/readme.html) ·
[Dataset and licence](https://dataspace.copernicus.eu/explore-data/data-collections/copernicus-contributing-missions/collections-description/COP-DEM).
This is a **digital surface model**, including reflective surfaces such as
vegetation; heights are metres relative to EGM2008.

Produced using Copernicus WorldDEM-30 © DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018 provided under COPERNICUS by the European Union and ESA; all rights reserved.

The committed `data/matterhorn_dem.npz` contains the small, unresampled source
crop, its coordinates, and source-tile SHA-256 hashes. Normal runs are offline
and need only NumPy and Plotly. Set `REFRESH_DATA = True` to rebuild it using
Rasterio (included in requirements.txt); full GeoTIFF tiles are cached in the
ignored project `.cache/terrain/` directory. Unlike HGT, these COG tiles have
already had shared south/east rows removed: **do not drop another seam row**.
Pixel-centre coordinates come from the GeoTIFF transform, not guessed bounds.
The 5 ? 5 km display extent is bilinearly sampled to a 281 ? 281 grid
for the browser (~18 m spacing).

The initially proposed Mapzen Skadi data read approximately 3,570 m at the summit
coordinate and put the crop maximum several hundred metres west, so this plot
uses Copernicus instead. The remaining difference from the independent summit
height is shown explicitly in the hover and exported page.

Run all cells from the repository or `notebooks/`. The export uses the shared
local Plotly bundle and legal footer. All terrain processing, plot controls,
and page styling live here.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import sys
import urllib.request
import numpy as np
import plotly.graph_objects as go

current = Path.cwd().resolve()
project_root = next((p for p in (current, *current.parents)
                     if (p / 'site_export.py').is_file() and (p / 'notebooks').is_dir()), None)
if project_root is None:
    raise RuntimeError('Run this notebook from the repository or notebooks directory.')
sys.path.insert(0, str(project_root)) if str(project_root) not in sys.path else None
from site_export import write_plot_page

SUMMIT_LAT, SUMMIT_LON = 45.97642, 7.65859
SUMMIT_M = 4478
RADIUS_KM, GRID_SIZE = 2.5, 281
REFRESH_DATA = False
data_file = project_root / 'data/matterhorn_dem.npz'
cache_dir = project_root / '.cache/terrain'
km_lat = 111.195
km_lon = km_lat * math.cos(math.radians(SUMMIT_LAT))
bounds = [SUMMIT_LAT - RADIUS_KM / km_lat, SUMMIT_LAT + RADIUS_KM / km_lat,
          SUMMIT_LON - RADIUS_KM / km_lon, SUMMIT_LON + RADIUS_KM / km_lon]

In [2]:
# Rebuild only on request or when the bundled crop is absent.
def download_tile(latitude):
    cache_dir.mkdir(parents=True, exist_ok=True)
    name = f'Copernicus_DSM_COG_10_N{latitude}_00_E007_00_DEM'
    path = cache_dir / f'{name}.tif'
    url = f'https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif'
    if not path.exists():
        print(f'Downloading {name} ...')
        request = urllib.request.Request(url, headers={'User-Agent': 'phey.app-terrain/1.0'})
        with urllib.request.urlopen(request, timeout=120) as response:
            raw = response.read()
        # Validate before promoting a download to the cache.
        with rasterio.io.MemoryFile(raw) as memory:
            with memory.open() as src:
                if src.shape != (3600, 3600) or src.crs.to_epsg() != 4326:
                    raise ValueError(f'Unexpected Copernicus tile: {name}')
        temporary = path.with_suffix('.part')
        temporary.write_bytes(raw)
        temporary.replace(path)
    return path, {'tile': name, 'url': url, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}

if REFRESH_DATA or not data_file.exists():
    import rasterio  # Only the optional data-refresh path needs GeoTIFF support.
    from rasterio.windows import Window
    s, n, w, e = bounds
    pieces, latitudes, sources = [], [], []
    source_lon = None
    for tile_lat in (46, 45):  # North to south, with no duplicated seam row.
        tile_path, source = download_tile(tile_lat)
        sources.append(source)
        with rasterio.open(tile_path) as src:
            t = src.transform
            if src.crs.to_epsg() != 4326 or src.shape != (3600, 3600):
                raise ValueError('Unexpected tile dimensions or coordinate system')
            if not (np.isclose(t.a, 1/3600) and np.isclose(t.e, -1/3600) and t.b == t.d == 0):
                raise ValueError('Unexpected tile spacing or rotation')
            first_lon, first_lat = t * (0.5, 0.5)
            c0, c1 = math.floor((w-first_lon)/t.a), math.ceil((e-first_lon)/t.a)+1
            r0 = max(0, math.floor((first_lat-n)/(-t.e)))
            r1 = min(src.height, math.ceil((first_lat-s)/(-t.e))+1)
            values = src.read(1, window=Window(c0, r0, c1-c0, r1-r0), masked=True)
            if np.ma.getmaskarray(values).any() or not np.isfinite(values).all():
                raise ValueError('Missing source elevations; refusing to invent terrain')
            pieces.append(values.data)
            latitudes.append(first_lat + np.arange(r0, r1)*t.e)
            tile_lon = first_lon + np.arange(c0, c1)*t.a
            if source_lon is not None and not np.allclose(source_lon, tile_lon):
                raise ValueError('Tile longitude grids do not align')
            source_lon = tile_lon
    source_lat = np.concatenate(latitudes)
    if not np.allclose(np.diff(source_lat), -1/3600, atol=1e-10, rtol=0):
        raise ValueError('Gap or duplicated row in the north/south tile join')
    crop = np.concatenate(pieces).astype(np.float32)
    provenance = json.dumps({'accessed': '2026-10-25', 'sources': sources,
        'dataset': 'Copernicus DEM GLO-30, AWS 2021 release',
        'format': 'GeoTIFF pixel centres, WGS84; elevations in metres, EGM2008',
        'attribution': 'Produced using Copernicus WorldDEM-30 © DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018 provided under COPERNICUS by the European Union and ESA; all rights reserved.'})
    data_file.parent.mkdir(exist_ok=True)
    np.savez_compressed(data_file, elevation_m=crop, latitude=source_lat,
                        longitude=source_lon, provenance=provenance)

with np.load(data_file, allow_pickle=False) as data:
    crop = data['elevation_m'].astype(np.float64)
    source_lat, source_lon = data['latitude'], data['longitude']
    provenance = json.loads(str(data['provenance']))
if provenance.get('dataset') != 'Copernicus DEM GLO-30, AWS 2021 release':
    raise ValueError('Unexpected cached dataset; rebuild with REFRESH_DATA = True')
assert crop.shape == (len(source_lat), len(source_lon))
assert np.all(np.diff(source_lat) < 0) and np.all(np.diff(source_lon) > 0)
if np.any(crop <= 0) or not np.isfinite(crop).all():
    raise ValueError('The Alpine crop contains invalid or missing elevations')
print(f'Loaded source crop: {crop.shape}; {data_file.stat().st_size / 1024:.0f} KiB')


Loaded source crop: (164, 235); 127 KiB


In [3]:
# Bilinear sampling preserves continuous geography across the 46°N tile seam.
lat = np.linspace(bounds[1], bounds[0], GRID_SIZE)
lon = np.linspace(bounds[2], bounds[3], GRID_SIZE)
if not (source_lat[-1] <= lat[-1] < lat[0] <= source_lat[0]
        and source_lon[0] <= lon[0] < lon[-1] <= source_lon[-1]):
    raise ValueError('Crop does not cover this extent. Refresh data for the new bounds.')

def sample_elevation(latitudes, longitudes):
    rows = (source_lat[0] - np.asarray(latitudes)) / (source_lat[0] - source_lat[1])
    cols = (np.asarray(longitudes) - source_lon[0]) / (source_lon[1] - source_lon[0])
    i = np.clip(np.floor(rows).astype(int), 0, crop.shape[0] - 2)
    j = np.clip(np.floor(cols).astype(int), 0, crop.shape[1] - 2)
    fy, fx = rows - i, cols - j
    return ((1-fy) * ((1-fx)*crop[i,j] + fx*crop[i,j+1])
            + fy * ((1-fx)*crop[i+1,j] + fx*crop[i+1,j+1]))

latitude, longitude = np.meshgrid(lat, lon, indexing='ij')
height_m = np.round(sample_elevation(latitude, longitude), 1)
x_km = np.round((lon - SUMMIT_LON) * km_lon, 5)
y_km = np.round((lat - SUMMIT_LAT) * km_lat, 5)
height_km = height_m / 1000
summit_dem = float(sample_elevation(SUMMIT_LAT, SUMMIT_LON))
assert height_m.shape == (GRID_SIZE, GRID_SIZE) and np.isfinite(height_m).all()
assert crop.min() <= height_m.min() + 0.1 and height_m.max() <= crop.max() + 0.1
assert 4000 < summit_dem < 4600, 'Unexpected summit elevation: check geographic indexing'
print(f'Elevation range: {height_m.min():,.0f}–{height_m.max():,.0f} m')
print(f'DEM at summit coordinate: {summit_dem:,.0f} m; independent summit reference: {SUMMIT_M:,} m')

Elevation range: 2,261–4,328 m
DEM at summit coordinate: 4,233 m; independent summit reference: 4,478 m


In [4]:
# All axes use km; aspectmode=data keeps the terrain physically proportioned.
BG, FG, MUTED = '#0d1117', '#eaf0f5', '#a4b2c1'
colors = [[0, '#263f45'], [.22, '#4c6a65'], [.43, '#8b9988'],
          [.60, '#aaa99b'], [.76, '#c0c3c2'], [.90, '#e0e8ed'], [1, '#fafcff']]
home_camera = dict(eye=dict(x=1.3, y=1.65, z=.85), up=dict(x=0, y=0, z=1),
                   center=dict(x=0, y=0, z=-.20), projection=dict(type='perspective'))
top_camera = dict(eye=dict(x=0, y=0, z=2.5), up=dict(x=0, y=1, z=0),
                  center=dict(x=0, y=0, z=0), projection=dict(type='orthographic'))
fig = go.Figure(go.Surface(
    x=x_km.astype(np.float32), y=y_km.astype(np.float32), z=height_km.astype(np.float32),
    surfacecolor=height_m.astype(np.float32), colorscale=colors, cmin=2000, cmax=4500,
    customdata=np.stack((np.round(latitude, 5), np.round(longitude, 5)), axis=-1).astype(np.float32),
    hovertemplate='Elevation %{z:.3f} km<br>%{customdata[0]:.5f}° N · %{customdata[1]:.5f}° E<extra>Terrain</extra>',
    colorbar=dict(title=dict(text='Elevation · m', side='top'), orientation='h',
                  x=.5, xanchor='center', y=-.08, len=.60, thickness=10,
                  tickvals=[2500,3000,3500,4000,4500], outlinewidth=0),
    contours=dict(z=dict(show=False, start=2.0, end=4.5, size=.1,
                         usecolormap=False, color='#45515c', width=1, highlight=False)),
    lighting=dict(ambient=.55, diffuse=.85, roughness=.9, specular=.15, fresnel=.15),
    lightposition=dict(x=10000, y=15000, z=20000), name='Terrain'))
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[SUMMIT_M/1000], mode='markers+text',
    marker=dict(size=4, color='#e9c58d', line=dict(width=1, color='#0d1117')),
    text=['Matterhorn · 4,478 m'], textposition='top center', textfont=dict(size=12, color=FG),
    hovertemplate=f'Summit reference: 4,478 m<br>DEM here: {summit_dem:,.0f} m<extra>Matterhorn</extra>',
    showlegend=False))
axis = dict(showbackground=False, gridcolor='#28323d', zeroline=False,
            tickfont=dict(size=10, color=MUTED), showspikes=False)
fig.update_layout(template='plotly_dark', paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family='Arial, sans-serif', color=FG, size=12), height=700,
    margin=dict(l=10, r=10, t=60, b=80), showlegend=False,
    scene=dict(bgcolor=BG, aspectmode='data', camera=home_camera,
        xaxis=dict(**axis, title='East · km', range=[-2.5,2.5], dtick=2.5),
        yaxis=dict(**axis, title='North · km', range=[-2.5,2.5], dtick=2.5),
        zaxis=dict(**axis, title='Altitude · km', range=[1.9,4.7], dtick=.5)),
    updatemenus=[dict(type='buttons', direction='right', x=0, xanchor='left', y=1.11,
        yanchor='top', bgcolor='#1b2733', bordercolor='#455569', font=dict(size=12),
        showactive=False, buttons=[
            dict(label='Reset 3D', method='relayout', args=[{'scene.camera':home_camera}]),
            dict(label='Overhead', method='relayout', args=[{'scene.camera':top_camera}]),
            dict(label='Contours', method='restyle', args=[{'contours.z.show':True}, [0]],
                 args2=[{'contours.z.show':False}, [0]])])])
config = dict(responsive=True, scrollZoom=True, displaylogo=False, displayModeBar=False)
fig.show(config=config)

In [5]:
# Deterministic export ID; page and mobile behaviour are part of the notebook.
output_file = project_root / 'output/20261004_parabola_3d.html'
plot_html = fig.to_html(include_plotlyjs=False, full_html=False, div_id='matterhorn-terrain',
    config=config, default_height='100%', post_script="""
const terrain = document.getElementById('matterhorn-terrain');
const desktopCamera = JSON.parse(JSON.stringify(terrain.layout.scene.camera));
const mobileCamera = JSON.parse(JSON.stringify(desktopCamera));
for (const axis of ['x', 'y', 'z']) mobileCamera.eye[axis] *= 1.5;
let wasCompact;
function keyboardControls() {
  terrain.querySelectorAll('.updatemenu-button').forEach(button => {
    button.setAttribute('role', 'button');
    button.setAttribute('tabindex', '0');
    button.setAttribute('aria-label', button.textContent);
    if (button.textContent === 'Contours') {
      button.setAttribute('aria-pressed', String(!!terrain.data[0].contours.z.show));
    }
    button.onkeydown = event => {
      if (event.key === 'Enter' || event.key === ' ') {
        event.preventDefault();
        button.dispatchEvent(new MouseEvent('click', {bubbles: true}));
      }
    };
  });
}
function fitTerrain() {
  const compact = window.innerWidth < 600;
  const home = compact ? mobileCamera : desktopCamera;
  const update = {height: compact ? 530 : Math.max(580, Math.min(800, window.innerHeight - 240)),
    'scene.xaxis.title.text': compact ? 'E ? km' : 'East ? km',
    'scene.yaxis.title.text': compact ? 'N ? km' : 'North ? km',
    'updatemenus[0].buttons[0].args': [{'scene.camera': home}]};
  if (compact !== wasCompact) update['scene.camera'] = home;
  wasCompact = compact;
  Plotly.relayout(terrain, update).then(keyboardControls);
}
terrain.on('plotly_afterplot', keyboardControls);
fitTerrain();
window.addEventListener('resize', fitTerrain);
""")
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Matterhorn 3D Topography | phey.app</title>
  <meta name="description" content="Explore the Matterhorn in three dimensions with real elevation data, a north-up overhead view and 100-metre contours.">
  <style>
    * {{ box-sizing: border-box; }}
    html, body {{ margin: 0; background: {BG}; color: {FG}; font-family: system-ui, sans-serif; }}
    main {{ max-width: 1600px; margin: auto; padding: 24px 32px 18px; }}
    a {{ color: #b8d2dd; text-underline-offset: 3px; }}
    .back {{ font-size: 13px; text-decoration: none; color: {MUTED}; }}
    .heading {{ display: flex; justify-content: space-between; align-items: end; gap: 20px; margin: 28px 0 16px; }}
    .eyebrow {{ color: #cbb48c; font-size: 11px; letter-spacing: .2em; text-transform: uppercase; margin: 0 0 8px; }}
    h1 {{ font-size: clamp(32px, 4vw, 52px); line-height: 1.05; margin: 0; font-weight: 500; letter-spacing: -.04em; }}
    .subtitle {{ color: {MUTED}; margin: 10px 0 0; font-size: 14px; }}
    .stats {{ display: flex; gap: 32px; flex-shrink: 0; }}
    .stats span {{ display: block; color: {MUTED}; font-size: 11px; margin-top: 6px; }}
    .stats strong {{ font-size: 20px; font-weight: 500; }}
    .plot {{ border-top: 1px solid #28323d; border-bottom: 1px solid #28323d; padding-top: 12px; }}
    .hint, .source {{ color: {MUTED}; font-size: 12px; line-height: 1.65; }}
    .hint {{ margin: 14px 0 0; }}
    details {{ margin-top: 16px; max-width: 100ch; }}
    summary {{ cursor: pointer; color: #cbd6df; }}
    .updatemenu-button:focus-visible rect {{ stroke: #e9c58d; stroke-width: 2px; }}
    a:focus-visible, summary:focus-visible {{ outline: 2px solid #e9c58d; outline-offset: 4px; }}
    @media(max-width: 600px) {{
      main {{ padding: 18px 12px; }}
      .heading {{ display: block; margin-top: 24px; }}
      .stats {{ gap: 24px; margin-top: 22px; }}
      .stats strong {{ font-size: 17px; }}
    }}
  </style>
</head>
<body>
<main>
  <a class="back" href="../index.html">← Back to phey.app</a>
  <header class="heading">
    <div><p class="eyebrow">Alpine terrain / Switzerland · Italy</p>
      <h1>Matterhorn</h1><p class="subtitle">A landscape in three dimensions.</p></div>
    <div class="stats" aria-label="Terrain facts">
      <div><strong>4,478 m</strong><span>Summit reference</span></div>
      <div><strong>5 ? 5 km</strong><span>Area explored</span></div>
      <div><strong>1 : 1</strong><span>True height scale</span></div>
    </div>
  </header>
  <div class="plot" role="region" aria-label="Interactive Matterhorn elevation model">{plot_html}</div>
  <p class="hint">Drag to orbit · Scroll or pinch to zoom · Overhead faces north · Contours every 100 m.<br>
    Colours show elevation, not snow cover. The initial view looks from the northeast, on the Zermatt side.</p>
  <details class="source"><summary>About the terrain &amp; data sources</summary>
    <p>A 5 ? 5 km crop of <a href="https://registry.opendata.aws/copernicus-dem/">Copernicus DEM GLO-30</a>,
    sampled from a 1 arc-second digital surface model (AWS 2021 release; heights relative to EGM2008). Horizontal distances use a local latitude-adjusted projection;
    all three axes use kilometres without vertical exaggeration.
    Terrain ranges from {height_m.min():,.0f} to {height_m.max():,.0f} m in this display.
    The summit marker is a separate reference: the elevation grid reads {summit_dem:,.0f} m at that coordinate.
    A gridded model smooths sharp peaks; this is not a navigation map.</p>
    <p>Produced using Copernicus WorldDEM-30 © DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018 provided under COPERNICUS by the European Union and ESA; all rights reserved.
    <a href="https://dataspace.copernicus.eu/explore-data/data-collections/copernicus-contributing-missions/collections-description/COP-DEM">Dataset &amp; licence</a> ·
    <a href="https://github.com/regnumproequo/p00/blob/main/notebooks/20261004_parabola_3d.ipynb">Notebook &amp; method</a>.
    Source tiles accessed 25 October 2026.</p>
  </details>
</main>
</body>
</html>"""
html = '\n'.join(line.rstrip() for line in html.splitlines()) + '\n'
write_plot_page(output_file, html)
print(f'Exported {output_file.name} ({output_file.stat().st_size / 1024**2:.2f} MiB)')

Exported 20261004_parabola_3d.html (1.73 MiB)
